# 07 · Tracing and eval

To compare three frameworks, every run has to be measured the same way: the same definition of a token,
an LLM call, a tool call and a second. This notebook connects Strands to the shared harness in
`agent_contracts.harness`, measures runs of the research workflow from notebook 05, and scores them with
the shared scoring, which is how the capstone's `run_eval.py` will work.

**You will learn**

- what the shared harness measures, and where each number comes from
- how a Strands hook feeds model calls into it
- how to run a small, scored evaluation and write it in the shared results format

In [1]:
from agent_contracts.harness import RunMetrics, build_results, verify_model
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report
from strands_agents_impl.llm import make_model
from strands_agents_impl.research_flow import run_research, to_report

verify_model()
model = make_model()
questions = {q["id"]: q for q in load_questions()}

## The shared definitions

| Number | Definition | Where it comes from |
|---|---|---|
| LLM calls | every model call in the run | the framework's callbacks, events or hooks |
| input and output tokens | summed over every model call | the counts Ollama reports with each response |
| tool calls | every attempted call, refused ones included | the shared tools record them (`research.tool_calls()`) |
| latency | wall clock for the whole question | `RunMetrics.measure()` |

Tool calls are counted inside the shared tools, so no framework can count them differently. Model calls
have to come from the framework, because only it sees them; in Strands that is a **hook**.

In [2]:
import inspect

import strands_agents_impl.metrics

print(inspect.getsource(strands_agents_impl.metrics.HarnessHook))

class HarnessHook(HookProvider):
    def __init__(self, metrics: RunMetrics) -> None:
        self.metrics = metrics
        self._seen: dict[int, tuple[int, int, int]] = {}  # per agent: cycles, input, output already counted

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(AfterInvocationEvent, self.after_invocation)

    def after_invocation(self, event: AfterInvocationEvent) -> None:
        m = event.agent.event_loop_metrics
        cycles = int(m.cycle_count)
        tokens_in, tokens_out = int(m.accumulated_usage.get("inputTokens", 0)), int(m.accumulated_usage.get("outputTokens", 0))
        seen_cycles, seen_in, seen_out = self._seen.get(id(event.agent), (0, 0, 0))
        self._seen[id(event.agent)] = (cycles, tokens_in, tokens_out)
        new_calls = cycles - seen_cycles
        if new_calls <= 0:
            return
        self.metrics.add_llm_call(tokens_in - seen_in, tokens_out - seen_out)  # first call carries the toke

Strands keeps running totals per agent (`agent.event_loop_metrics`) but updates the token counts only
after a model call's events have fired, so a hook that read them after every call would always be one call
behind. This one settles at the end of each agent invocation instead, which gives exact totals and call
counts. The research workflow's structured steps are single model calls made outside any agent
(notebook 05), so `structured_call` records those itself.

## One measured run

In [3]:
def run_question(qid: str) -> dict:
    """Run one question through the workflow, measured and scored the shared way."""
    q = questions[qid]
    research.reset()
    metrics = RunMetrics()
    with metrics.measure():
        state = run_research(qid, q["question"], model, metrics=metrics)
    report = to_report(state)
    scored = score_report(report, q)
    return {"id": qid, "passed": scored["passed"], "scores": scored["scores"], "problems": scored["problems"],
            "report": report.model_dump(), "tool_calls": research.tool_calls(),
            "budget_exhausted": research.budget_exhausted(), **metrics.as_dict(), "error": None}


item = run_question("Q07_deployment")
print({k: item[k] for k in ("passed", "llm_calls", "input_tokens", "output_tokens", "latency_s", "tool_calls")})
print("answer:", item["report"]["answer"])
print("scores:", item["scores"], item["problems"])

{'passed': False, 'llm_calls': 8, 'input_tokens': 10870, 'output_tokens': 461, 'latency_s': 195.748, 'tool_calls': ['web_search', 'web_search', 'web_search', 'web_search']}
answer: Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025 [F04]. Earlier reports from 2023 indicated that TidePure units were operational in 9 provinces [F03].
scores: {'fact_recall': 1.0, 'facts_missing': [], 'contradiction_free': False, 'citations_valid': True} ["repeats stale or wrong figures: ['9 provinces']"]


Q07 asks how many units Veltara has deployed and in how many provinces. The 2023 article still says
9 provinces; the current figure is 14. Mentioning 9 at all fails `contradiction_free`.

## A small eval

An evaluation is the same thing over a fixed set of cases, scored automatically, so you can re-run it after
every change and see whether things got better. These are the same questions the LangGraph and CrewAI
tutorials use: a calculation (Q03), a recall with a distractor article about another company (Q08), and the
Series B round (Q04).

In [4]:
items = [item]
for qid in ("Q03_revenue_growth", "Q08_recall", "Q04_series_b"):
    items.append(run_question(qid))

print(f"{'question':20} {'pass':5} {'recall':6} {'calls':5} {'tokens':>7} {'secs':>5}  tools")
for it in items:
    print(f"{it['id']:20} {str(it['passed']):5} {it['scores']['fact_recall']:6} {it['llm_calls']:5} "
          f"{it['input_tokens'] + it['output_tokens']:7} {it['latency_s']:5.0f}  {it['tool_calls']}")
for it in items:
    if it["problems"]:
        print(f"\n{it['id']}: {it['problems']}\n  answer: {it['report']['answer']}")

question             pass  recall calls  tokens  secs  tools
Q07_deployment       False    1.0     8   11331   196  ['web_search', 'web_search', 'web_search', 'web_search']
Q03_revenue_growth   True     1.0     6    6199   118  ['web_search', 'web_search', 'calculator']
Q08_recall           True     1.0     5    2978    79  ['web_search']
Q04_series_b         True     1.0     5    3162    97  ['web_search']

Q07_deployment: ["repeats stale or wrong figures: ['9 provinces']"]
  answer: Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025 [F04]. Earlier reports from 2023 indicated that TidePure units were operational in 9 provinces [F03].


In the saved run three of four passed. Q07 failed: the answer gave the current 14 provinces, then added
that "earlier reports from 2023 indicated ... 9 provinces", citing the stale article. The critic passed
it; the plain-text check did not. That is the same failure LangGraph's Q01 showed in notebook 05: a model
that knows which figure is current still mentions the old one, and a model acting as critic lets it
through.

Across the three tutorials, the same four questions with the same instructions and model:

| | LangGraph | CrewAI | Strands |
|---|---|---|---|
| passed | 4 of 4 | 4 of 4 | 3 of 4 |
| total tokens | 15,669 | 23,847 | 23,670 |
| total time | 310 s | 440 s | 490 s |
| web searches | 5 | 8 | 8 |

These are single runs of four questions on one laptop, so treat the differences as first signals, not a
verdict; the capstone evaluations run every question in every framework.

## The shared results format

`build_results` wraps the items in the format every framework writes, and computes the summary the same way
for all of them. The capstone's `run_eval.py` writes it to `results/research/strands.json`; this notebook
only prints it, so a tutorial run never overwrites real results.

In [5]:
from importlib.metadata import version

from agent_contracts.harness import load_settings

results = build_results("research", "strands", version("strands-agents"), load_settings().model, items)
print("model:", results["model"])
print("summary:", results["summary"])
print("keys per question:", sorted(results["questions"][0]))

model: qwen3-agents
summary: {'passed': 3, 'total': 4, 'total_tokens': 23670, 'total_latency_s': 489.63}
keys per question: ['budget_exhausted', 'error', 'id', 'input_tokens', 'latency_s', 'llm_calls', 'output_tokens', 'passed', 'problems', 'report', 'scores', 'tool_calls']


## Seeing inside a run

Numbers say *how much*; traces say *what happened*. In Strands the simplest trace is the conversation
itself, `agent.messages`, with every `toolUse` and `toolResult` (notebook 02), and the hooks used
throughout this tutorial can log any event in the loop. Strands also emits OpenTelemetry traces, which you
can send to a collector you run; nothing is sent anywhere unless you configure an exporter.

## How the others do this

- **LangGraph** (LangChain) passes a callback handler in the run's `config`; its `on_llm_end` sees every
  model call as it happens, nested agents included.
- **CrewAI** publishes an event on its event bus after every model call; a listener collects them, and
  the bus must be flushed before reading, because events arrive asynchronously.

All three feed the same `RunMetrics`, and all three count tool calls through the shared tools.

## Takeaways

- Fix the definitions once, in shared code, then connect each framework to them.
- Count what you can outside the framework (tool calls); take only model calls from the framework.
- In Strands, settle a hook's numbers when an invocation ends, not after each call.
- An eval is a fixed set of cases, scored automatically, re-run after every change.
- That completes the components. The capstones build ShipFast and the research agent in full.